# DPO data preparation — original datasets

This version works directly from files named `D_syn_COUNTRYNAME.jsonl` (for example, `D_syn_USA.jsonl`). It:

1. normalizes prompts by adding/fixing the `Context:` prefix;
2. renames GPS dimensions exactly as in `preprocessing_without_gpu_multi_country.ipynb`;
3. validates the DPO fields `prompt`, `chosen`, and `rejected`;
4. creates train/eval files without generating the 3,594-row extended dataset; and
5. precomputes reference-model log probabilities for the training split.


In [1]:
!pip -q install -U "transformers>=4.41.0" "datasets>=2.18.0" "accelerate>=0.30.0" \
                 "trl>=0.11.0" "peft>=0.11.1" "bitsandbytes>=0.46.1" "safetensors>=0.4.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.8 MB/s eta 0:00:00


In [2]:
# Hugging Face login is needed for gated Llama weights.
from huggingface_hub import notebook_login
notebook_login()

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

# Countries to prepare. Use the short tag used for the adapter folder.
# US is special-cased to read D_syn_USA.jsonl.
COUNTRY_CODES = ["RUS"]

# Add special filename mappings here if another country's raw filename does not
# equal its short country tag. Examples:
# COUNTRY_NAME_MAP = {"US": "USA", "MX": "MEX"}
COUNTRY_NAME_MAP = {
    "US": "USA",
}

def resolve_country_codes(country_code):
    data_country_code = COUNTRY_NAME_MAP.get(country_code, country_code)
    adapter_tag = country_code
    return data_country_code, adapter_tag

# Raw D_syn_* files live here.
RAW_DATA_DIR = Path("/content/drive/MyDrive/DPO/pre-processing")

# Train/eval/precomputed files are written here.
DATA_DIR = Path("/content/drive/MyDrive/DPO")
DATA_DIR.mkdir(parents=True, exist_ok=True)


def build_country_paths(country_code):
    data_country_code, adapter_tag = resolve_country_codes(country_code)

    raw_file = RAW_DATA_DIR / f"D_syn_{data_country_code}.jsonl"
    train_file = DATA_DIR / f"{data_country_code}_train.jsonl"
    eval_file = DATA_DIR / f"{data_country_code}_eval.jsonl"
    out_file = DATA_DIR / f"{data_country_code}_train_with_ref.jsonl"

    return raw_file, train_file, eval_file, out_file, data_country_code, adapter_tag

for country_code in COUNTRY_CODES:
    print(country_code, "->", build_country_paths(country_code))

RUS -> (PosixPath('/content/drive/MyDrive/DPO/pre-processing/D_syn_RUS.jsonl'), PosixPath('/content/drive/MyDrive/DPO/RUS_train.jsonl'), PosixPath('/content/drive/MyDrive/DPO/RUS_eval.jsonl'), PosixPath('/content/drive/MyDrive/DPO/RUS_train_with_ref.jsonl'), 'RUS', 'RUS')


In [5]:
import os

# Set before importing torch/transformers/accelerate.
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TRANSFORMERS_NO_BF16"] = "1"

import torch

torch.set_default_dtype(torch.float16)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU for reference-logprob precomputation.")

CUDA: 12.8
GPU: Tesla T4


In [6]:
import json
import random
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
TRAIN_FRAC = 0.80
SEED = 42

# Same renames as preprocessing_without_gpu_multi_country.ipynb.
DIMENSION_RENAMES = {
    "negrecip": "negative_reciprocity",
    "posrecip": "positive_reciprocity",
    "risktaking": "risk_taking",
}

REQUIRED_DPO_FIELDS = ("prompt", "chosen", "rejected")


def normalize_prompt(text):
    if not isinstance(text, str) or not text.strip():
        return text

    text = text.strip()
    first = text.split()[0]

    if first == "Context:":
        return text
    if first == "context:":
        return text[:1].upper() + text[1:]
    return "Context: " + text


def normalize_row(row):
    row = dict(row)
    row["prompt"] = normalize_prompt(row.get("prompt"))

    if "gps_dimension" in row:
        row["gps_dimension"] = DIMENSION_RENAMES.get(
            row["gps_dimension"], row["gps_dimension"]
        )
    return row


def validate_rows(rows, source_name="dataset"):
    if not rows:
        raise ValueError(f"{source_name} is empty")

    for i, row in enumerate(rows):
        for field in REQUIRED_DPO_FIELDS:
            value = row.get(field)
            if not isinstance(value, str) or not value.strip():
                raise ValueError(
                    f"{source_name}: row {i} has missing/empty required field {field!r}"
                )
        if row["chosen"].strip() == row["rejected"].strip():
            raise ValueError(f"{source_name}: row {i} has identical chosen and rejected")


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def split_country_file(raw_path, train_path, eval_path, country, train_frac=0.80, seed=42):
    rows = [normalize_row(row) for row in load_jsonl(raw_path)]
    validate_rows(rows, source_name=str(raw_path))

    # Add stable metadata. Existing values are preserved.
    for i, row in enumerate(rows):
        row.setdefault("country", country)
        row.setdefault("item_id", f"{country}_{i:04d}")

    # Group by prompt so repeated versions of the same scenario can never leak
    # across train/eval. For the original files, prompts are normally unique.
    groups = {}
    for row in rows:
        groups.setdefault(row["prompt"], []).append(row)

    group_keys = list(groups)
    rng = random.Random(seed)
    rng.shuffle(group_keys)

    n_train_groups = int(len(group_keys) * train_frac)
    train_keys = set(group_keys[:n_train_groups])
    eval_keys = set(group_keys[n_train_groups:])

    train_rows = [row for key in group_keys if key in train_keys for row in groups[key]]
    eval_rows = [row for key in group_keys if key in eval_keys for row in groups[key]]

    train_prompts = {row["prompt"] for row in train_rows}
    eval_prompts = {row["prompt"] for row in eval_rows}
    assert train_prompts.isdisjoint(eval_prompts), "Prompt leakage between train and eval"

    write_jsonl(train_rows, train_path)
    write_jsonl(eval_rows, eval_path)

    print(f"{country}: {len(rows)} total rows / {len(group_keys)} unique prompts")
    print(f"  train: {len(train_rows)} rows -> {train_path}")
    print(f"  eval:  {len(eval_rows)} rows -> {eval_path}")

    renamed_dims = sorted({row.get("gps_dimension") for row in rows if row.get("gps_dimension")})
    print("  normalized dimensions:", renamed_dims)
    print("  sample prompt:", rows[0]["prompt"][:120], "...")

    return train_rows, eval_rows

In [7]:
# Normalize and split every requested country.
prepared_splits = {}

for country_code in COUNTRY_CODES:
    raw_file, train_file, eval_file, out_file, data_country_code, adapter_tag = build_country_paths(country_code)

    if not raw_file.exists():
        raise FileNotFoundError(
            f"Raw file not found: {raw_file}. Expected a file named "
            f"D_syn_{data_country_code}.jsonl"
        )

    train_rows, eval_rows = split_country_file(
        raw_path=raw_file,
        train_path=train_file,
        eval_path=eval_file,
        country=data_country_code,
        train_frac=TRAIN_FRAC,
        seed=SEED,
    )
    prepared_splits[country_code] = (train_rows, eval_rows, out_file)

RUS: 658 total rows / 658 unique prompts
  train: 526 rows -> /content/drive/MyDrive/DPO/RUS_train.jsonl
  eval:  132 rows -> /content/drive/MyDrive/DPO/RUS_eval.jsonl
  normalized dimensions: ['altruism', 'negative_reciprocity', 'patience', 'positive_reciprocity', 'risk_taking', 'trust']
  sample prompt: Context: A fellow student tutored me for free before a critical exam, and now they're facing a tight deadline for their  ...


In [8]:
# Reference-model setup.
MAX_PROMPT_TOKENS = 256
MAX_COMPLETION_TOKENS = 256
compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading reference (base) model in 4-bit...")
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
)
ref_model.eval()


def build_user_prompt(prompt_text: str) -> str:
    return (
        "You are answering a questionnaire as an individual person. "
        "Respond naturally and thoughtfully, as someone would in real life. "
        "Do not mention being an AI or assistant. "
        "Keep the answer short, under 3 sentences. "
        "Give a sincere, human-like answer.\n\n"
        "Situation:\n"
        f"{prompt_text.strip()}\n\n"
        "Answer:"
    )


def format_prompt_text(tokenizer, prompt_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": build_user_prompt(prompt_text)}],
        tokenize=False,
        add_generation_prompt=True,
    )


@torch.no_grad()
def seq_logprob_for_completion(prompt_text: str, completion_text: str) -> float:
    """Sum log p(completion | prompt) under the base/reference model."""
    prompt = format_prompt_text(tokenizer, prompt_text)

    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
    if len(prompt_ids) > MAX_PROMPT_TOKENS:
        prompt_ids = prompt_ids[-MAX_PROMPT_TOKENS:]

    comp_ids = tokenizer(completion_text, add_special_tokens=False).input_ids
    if len(comp_ids) > MAX_COMPLETION_TOKENS:
        comp_ids = comp_ids[:MAX_COMPLETION_TOKENS]

    input_ids = prompt_ids + comp_ids
    if len(input_ids) < 2 or not comp_ids:
        return float("-inf")

    labels = ([-100] * len(prompt_ids)) + comp_ids
    input_ids_t = torch.tensor([input_ids], device=ref_model.device)
    labels_t = torch.tensor([labels], device=ref_model.device)

    logits = ref_model(input_ids=input_ids_t).logits
    log_probs = torch.log_softmax(logits, dim=-1)[:, :-1, :]
    labels_shifted = labels_t[:, 1:]

    mask = labels_shifted.ne(-100)
    gold = labels_shifted.clone()
    gold[~mask] = 0
    token_logps = log_probs.gather(-1, gold.unsqueeze(-1)).squeeze(-1)
    token_logps = token_logps * mask

    return float(token_logps.sum().detach().cpu().to(torch.float32))

Loading tokenizer...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [1]:
# Precompute reference log-probabilities for each country's TRAIN split only.
for country_code in COUNTRY_CODES:
    train_rows, _, out_file = prepared_splits[country_code]

    print(f"\nPrecomputing {country_code}: {len(train_rows)} training examples")
    with open(out_file, "w", encoding="utf-8") as out:
        for i, ex in enumerate(train_rows, 1):
            ex = dict(ex)
            ex["ref_chosen_logps"] = seq_logprob_for_completion(ex["prompt"], ex["chosen"])
            ex["ref_rejected_logps"] = seq_logprob_for_completion(ex["prompt"], ex["rejected"])
            out.write(json.dumps(ex, ensure_ascii=False) + "\n")

            if i % 25 == 0 or i == len(train_rows):
                print(f"  precomputed {i}/{len(train_rows)}")

    print("Done. Wrote:", out_file)

# Free VRAM before training in the next notebook.
del ref_model
torch.cuda.empty_cache()

NameError: name 'COUNTRY_CODES' is not defined